In [ ]:
# ── Model selection ─────────────────────────────────────────────────────────
model_arm = 'qualitative'   # qualitative | ms | ic50_500 | ic50_1000
n_rows = None               # None = the whole shipped set (48,352 rows); set e.g. 2000 to sample it
# ────────────────────────────────────────────────────────────────────────────
# This demo runs the fp16 path and encodes every epitope with ESMC at run time (with flash-attn
# off on pre-Ampere GPUs and on CPU), so its scores move in the last decimals and do not
# reproduce the paper's numbers bit for bit.

import os
from huggingface_hub import hf_hub_download

REPO = os.path.abspath('..')            # this notebook lives in demo/
MODELS = os.path.join(REPO, 'models')   # where configs/predict/*.py look for weights

ARMS = {  # arm -> (predict config, float16 checkpoint on daylight-00/prepibind-demo)
    'qualitative': ('config_demo.py',      'prepibind_qualitative_s100_f0_fp16.pt'),
    'ms':          ('config_ms.py',        'prepibind_ms_s128_f3_fp16.pt'),
    'ic50_500':    ('config_ic50_500.py',  'prepibind_ic50_500_s128_f2_fp16.pt'),
    'ic50_1000':   ('config_ic50_1000.py', 'prepibind_ic50_1000_s42_f1_fp16.pt'),
}
config_name, chkp_name = ARMS[model_arm]
config_path = os.path.join(REPO, 'configs', 'predict', config_name)

# The ESMC backbone plus the one head. Every released checkpoint here is float16 weights only --
# see demo/build_demo_assets.py. Use the paths hf_hub_download returns.
esm_chkp_path = hf_hub_download('daylight-00/esmc-300m-2024-12', 'esmc_300m_2024_12_v0_fp16.pth', local_dir=MODELS)
chkp_path = hf_hub_download('daylight-00/prepibind-demo', chkp_name, local_dir=MODELS)

In [ ]:
import pandas as pd
from prepibind.inference import main as inference, load_config

config = load_config(config_path, out_path='outputs',
                     chkp_path=chkp_path, esm_chkp_path=esm_chkp_path)
if n_rows is not None:
    os.makedirs('outputs', exist_ok=True)
    head_path = 'outputs/dataset_head.csv'
    pd.read_csv(config['Data']['test_path']).head(n_rows).to_csv(head_path, index=False)
    config['Data']['test_path'] = head_path

df = inference(config)
df.head()

Results land in `demo/outputs/prediction.csv`, plus `demo/outputs/plot.png` when the config's
`plot` is on.

The default test set is the whole shipped `demo/data/dataset_demo.csv` — 48,352 rows, every
epitope encoded with ESMC at run time, which is hours on CPU. Set `n_rows` in the first cell to
run a slice instead; it is written to `demo/outputs/dataset_head.csv` and used as the test set.

Allele names must exist in `demo/data/mhc_mapping_demo.csv` (116 alleles), and the `MHC` column
pairs them as `beta_alpha`, e.g. `HLA-DRB1*01:01_HLA-DRA*01:01`.